In [ ]:
#%%capture
!wget "https://repo1.maven.org/maven2/org/apache/spark/spark-sql-kafka-0-10_2.12/3.5.0/spark-sql-kafka-0-10_2.12-3.5.0.jar"
!wget "https://repo1.maven.org/maven2/org/apache/spark/spark-streaming-kafka-0-10_2.12/3.5.0/spark-streaming-kafka-0-10_2.12-3.5.0.jar"

In [1]:
import os
os.environ['PYSPARK_SUBMIT_ARGS'] = (
    '--packages '
    'org.apache.spark:spark-streaming-kafka-0-10_2.12:3.5.0,'
    'org.apache.spark:spark-sql-kafka-0-10_2.12:3.5.0 '
    'pyspark-shell'
)

In [2]:
!pip install mistralai==1.9.7

In [3]:
import json
import base64
import time
from pyspark.sql import SparkSession, Row
from pyspark.sql.functions import col
from pyspark.sql.types import StructType, StructField, StringType, BinaryType
from mistralai import Mistral, ImageURLChunk, TextChunk

spark = SparkSession.builder.appName("OCR").getOrCreate()
spark.sparkContext.setLogLevel("WARN")

In [4]:
# 1. Read Stream from Kafka
KAFKA_BROKER = os.environ.get("KAFKA_BROKER", "192.168.1.124:8097")
KAFKA_TOPIC = os.environ.get("KAFKA_TOPIC", "input")

raw_stream = (
    spark.readStream
    .format("kafka")
    .option("kafka.bootstrap.servers", KAFKA_BROKER)
    .option("subscribe", KAFKA_TOPIC)
    .option("startingOffsets", "latest")
    .load()
)

raw_stream.printSchema()

root
 |-- key: binary (nullable = true)
 |-- value: binary (nullable = true)
 |-- topic: string (nullable = true)
 |-- partition: integer (nullable = true)
 |-- offset: long (nullable = true)
 |-- timestamp: timestamp (nullable = true)
 |-- timestampType: integer (nullable = true)



In [5]:
# 2. Prepare data - extract binary image and filename
processed_data = raw_stream.select(
    col("value").cast("BINARY").alias("png_data"),
    col("key").cast(StringType()).alias("filename")
)

In [6]:
# 3. Initialize Mistral client
api_key = os.environ.get("MISTRAL_API_KEY", "BsRLt6Eb3PbKlnifzzKyhLC7QdBqLIBj")
client = Mistral(api_key=api_key)


def call_mistral_ocr(row: Row):
    """Process image with Mistral OCR and return structured JSON with token usage."""
    try:
        filename = row['filename']
        png_bytes = row['png_data']

        # Encode image as base64 for API
        encoded = base64.b64encode(png_bytes).decode()
        base64_data_url = f"data:image/jpeg;base64,{encoded}"

        # Step 1: OCR extraction
        image_response = client.ocr.process(
            document=ImageURLChunk(image_url=base64_data_url),
            model="mistral-ocr-latest"
        )
        image_ocr_markdown = image_response.pages[0].markdown

        # Step 2: Convert to structured JSON
        chat_response = client.chat.complete(
            model="pixtral-12b-latest",
            messages=[
                {
                    "role": "user",
                    "content": [
                        ImageURLChunk(image_url=base64_data_url),
                        TextChunk(
                            text=(
                                f"This is image's OCR in markdown:\n\n{image_ocr_markdown}\n.\n"
                                "Convert this into a sensible structured json response. "
                                "The output should be strictly be json with no extra commentary"
                            )
                        ),
                    ],
                }
            ],
            response_format={"type": "json_object"},
            temperature=0,
        )

        usage = chat_response.usage

        response_dict = json.loads(chat_response.choices[0].message.content)
        return filename, json.dumps(response_dict), usage.prompt_tokens, usage.completion_tokens, usage.total_tokens

    except Exception as e:
        print(f"Error processing {row.get('filename', 'unknown')}: {e}")
        return row.get('filename', 'unknown'), json.dumps({"error": str(e)}), 0, 0, 0


In [7]:
def process_batch(df, batch_id):
    """Process each micro-batch: run OCR and write results to JSON."""
    start_time_full = time.time()
    total_ai_duration = 0.0
    total_input_tokens = 0
    total_output_tokens = 0
    total_tokens = 0

    # ⚠️ WARNING: .collect() brings ALL data to the driver program.
    # Use for testing/low-volume only. For high-volume, consider a custom Kafka Connect Sink.
    rows = df.collect()
    results = []

    for row in rows:
        start_time_ai = time.time()
        filename, ocr_text, input_tok, output_tok, total_tok = call_mistral_ocr(row)
        total_ai_duration += (time.time() - start_time_ai)

        total_input_tokens += input_tok
        total_output_tokens += output_tok
        total_tokens += total_tok

        if filename is None:
            filename = f"image_{batch_id}"
        results.append((filename, ocr_text))

    if results:
        schema = StructType([
            StructField("filename", StringType(), True),
            StructField("ocr_text", StringType(), True)
        ])

        results_df = df.sparkSession.createDataFrame(results, schema)

        # Write results to JSON files
        output_dir = os.environ.get("OCR_OUTPUT_DIR", "/home/jovyan/json")
        batch_output_path = os.path.join(output_dir, f"batch_{batch_id}")

        results_df.write \
            .format("json") \
            .mode("overwrite") \
            .save(batch_output_path)

        total_duration = time.time() - start_time_full
        time_usage = total_duration - total_ai_duration

        print(f"Batch {batch_id} complete: {batch_output_path}")
        print(f"  Timing   - Total: {total_duration:.2f}s | AI: {total_ai_duration:.2f}s | Other: {time_usage:.2f}s")
        print(f"  Tokens   - Input: {total_input_tokens} | Output: {total_output_tokens} | Total: {total_tokens}")


In [8]:
# 4. Install evaluation dependencies
!pip install jiwer

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 110.5/110.5 kB 538.1 kB/s eta 0:00:00a 0:00:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.2/3.2 MB 704.1 kB/s eta 0:00:0000:0100:01
  Attempting uninstall: click
    Found existing installation: click 8.1.7
    Uninstalling click-8.1.7:
      Successfully uninstalled click-8.1.7


In [9]:
from jiwer import cer, wer


def evaluate_ocr_accuracy(ground_truth: str, ocr_text: str):
    """Calculate CER and WER between ground truth and OCR output."""
    character_error_rate = cer(ground_truth, ocr_text)
    word_error_rate = wer(ground_truth, ocr_text)

    print(f"CER: {character_error_rate:.4f} ({character_error_rate*100:.2f}%)")
    print(f"WER: {word_error_rate:.4f} ({word_error_rate*100:.2f}%)")
    return character_error_rate, word_error_rate


def evaluate_structured_json(expected: dict, actual: dict, parent_key=""):
    """Recursively compare structured JSON fields."""
    results = {}

    for key in expected:
        full_key = f"{parent_key}.{key}" if parent_key else key

        if key not in actual:
            results[full_key] = "✗ Missing in output"
            continue

        if isinstance(expected[key], dict) and isinstance(actual.get(key), dict):
            results.update(evaluate_structured_json(expected[key], actual[key], full_key))
        elif expected[key] == actual[key]:
            results[full_key] = "✓ Match"
        else:
            results[full_key] = f"✗ Mismatch: expected={expected[key]}, got={actual[key]}"

    return results


def test_single_image(image_path: str, ground_truth_path: str = None):
    """Test OCR accuracy on a single image and display token usage."""
    if not os.path.exists(image_path):
        print(f"Image not found: {image_path}")
        return None

    # Read image
    with open(image_path, "rb") as f:
        png_bytes = f.read()

    # Create test row
    test_row = Row(filename=os.path.basename(image_path), png_data=png_bytes)

    # Run OCR
    print(f"\n{'='*50}")
    print(f"Processing: {image_path}")
    print(f"{'='*50}")

    filename, ocr_json, input_tokens, output_tokens, total_tokens = call_mistral_ocr(test_row)
    ocr_data = json.loads(ocr_json)

    # Display Token Usage
    print(f"\n{'='*50}")
    print("TOKEN USAGE")
    print(f"{'='*50}")
    print(f"  Input tokens:  {input_tokens}")
    print(f"  Output tokens: {output_tokens}")
    print(f"  Total tokens:  {total_tokens}")

    # Display OCR Result
    print(f"\n{'='*50}")
    print("OCR RESULT")
    print(f"{'='*50}")
    print(json.dumps(ocr_data, indent=2, ensure_ascii=False))

    # Evaluate if ground truth exists
    if ground_truth_path and os.path.exists(ground_truth_path):
        with open(ground_truth_path, 'r', encoding='utf-8') as f:
            ground_truth = json.load(f) if ground_truth_path.endswith('.json') else f.read().strip()

        print(f"\n{'='*50}")
        print("ACCURACY EVALUATION")
        print(f"{'='*50}")

        if isinstance(ground_truth, dict):
            results = evaluate_structured_json(ground_truth, ocr_data)
            for key, status in results.items():
                print(f"  {key}: {status}")
        else:
            evaluate_ocr_accuracy(ground_truth, json.dumps(ocr_data, ensure_ascii=False))
    else:
        print(f"\nNo ground truth provided.")
        print("Create ground_truth.json or ground_truth.txt to enable accuracy evaluation.")

    return ocr_data


def create_ground_truth_template(image_path: str):
    """Create a template ground truth file for manual annotation."""
    template = {
        "summary": {
            "course": "<expected course name>",
            "semester": "<expected semester>",
            "student_info": {
                "name": "<expected name>",
                "year": "<expected year>",
                "registration_number": "<expected reg #>",
                "group": "<expected group>"
            },
            "assessment_time": "<expected time>"
        },
        "questions": [
            {"question": "<expected question 1>"},
            {"question": "<expected question 2>"}
        ]
    }

    template_path = "ground_truth.json"
    with open(template_path, 'w', encoding='utf-8') as f:
        json.dump(template, f, indent=2, ensure_ascii=False)

    print(f"Created template: {template_path}")
    print("Edit this file with the actual expected values from the image.")


In [10]:
# 5. Test OCR on sample image (receipt1.png)
# First create ground truth template, then edit it with actual values

create_ground_truth_template("receipt1.png")

# After editing ground_truth.json, run:
# ocr_result = test_single_image("receipt1.png", "ground_truth.json")

Created template: ground_truth.json
Edit this file with the actual expected values from the image.


In [ ]:
# 6. Start streaming query
query = (
    processed_data.writeStream
    .foreachBatch(process_batch)
    .start()
)

query.awaitTermination()

Batch 1 complete: /home/jovyan/json/batch_1
  Timing   - Total: 14.39s | AI: 6.35s | Other: 8.04s
  Tokens   - Input: 1591 | Output: 325 | Total: 1916
Batch 2 complete: /home/jovyan/json/batch_2
  Timing   - Total: 15.15s | AI: 12.20s | Other: 2.95s
  Tokens   - Input: 1591 | Output: 325 | Total: 1916
Batch 3 complete: /home/jovyan/json/batch_3
  Timing   - Total: 13.92s | AI: 7.63s | Other: 6.29s
  Tokens   - Input: 1591 | Output: 325 | Total: 1916
Batch 4 complete: /home/jovyan/json/batch_4
  Timing   - Total: 9.31s | AI: 6.29s | Other: 3.02s
  Tokens   - Input: 1591 | Output: 334 | Total: 1925
Batch 5 complete: /home/jovyan/json/batch_5
  Timing   - Total: 10.97s | AI: 8.49s | Other: 2.48s
  Tokens   - Input: 1591 | Output: 325 | Total: 1916
Batch 6 complete: /home/jovyan/json/batch_6
  Timing   - Total: 11.26s | AI: 7.19s | Other: 4.07s
  Tokens   - Input: 1591 | Output: 336 | Total: 1927
Batch 7 complete: /home/jovyan/json/batch_7
  Timing   - Total: 11.94s | AI: 9.17s | Other: 2.

In [ ]:
# Stop all active streams
for q in spark.streams.active:
    q.stop()